# Changing the Interval: the Two Dials

A confidence interval has exactly two dials. Turn up the **sample size** and the sampling distribution of $\bar{X}$ gets taller and skinnier ($\sigma_{\bar{X}} = \sigma/\sqrt{n}$), so the same 95% of sample means fits inside smaller margins of error — a narrower interval at the *same* confidence. Turn down the **confidence level** (95% → 70%) and the margins also shrink — but now the recipe captures $\mu$ only 70% of the time. One dial buys precision with data; the other buys it with risk. It is based on the second video of Lesson 7 (Confidence Intervals).

## 1. Introduction

The last notebook built a 95% confidence interval from a sample of one person — with a margin of error of nearly $2\sigma$, a lot of road to search. This notebook asks how to do better, and finds the answer in a fact from Lesson 5: the sampling distribution of the mean narrows as the sample grows. It also examines the other, cheaper-looking dial — lowering the confidence level — and why that one is usually the wrong knob to turn.

After completing this notebook you will be able to:

- Recall why $\mathbb{E}[\bar{X}] = \mu$ for every sample size while $\sigma_{\bar{X}} = \sigma/\sqrt{n}$ shrinks with $n$.
- Explain why a taller, skinnier sampling distribution needs smaller margins of error at the same confidence level.
- Show what lowering the confidence level does — and does not — change about the intervals.
- Compute the margin of error $z \cdot \sigma/\sqrt{n}$ for any sample size and confidence level, and invert it to find the $n$ a target width requires.
- Argue why "collect more data" is the honest way to a narrow interval, and why levels below 90% are rare.

## 2. Theory

### Recap: the distribution of $\bar{X}$

Two facts from Lesson 5 about the sampling distribution of the sample mean:

$$
\mu_{\bar{X}} = \mu \qquad \text{(for every sample size } n\text{)}, \qquad\qquad
\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{n}} \qquad \text{(shrinks with } n\text{)}
$$

With $n = 1$ the two distributions coincide — $\bar{X} \sim N(\mu, \sigma^2)$, which is why the last notebook could use the population curve directly.

### Dial 1: the sample size

Increase $n$ from 1 to 2 and the curve of $\bar{X}$ keeps its center but its standard deviation drops to $\sigma/\sqrt{2}$: the curve **doesn't move left or right, it gets taller and skinnier**. Outcomes now concentrate more tightly around $\mu$ — so capturing 95% of all sample means needs a *smaller* margin of error on each side. Push on to $n = 10$ and the margins shrink again.

Generate many intervals at $n = 2$ and many at $n = 10$, both at the 95% level: roughly 95% of each batch contains $\mu$ and about 5% miss. **Same confidence — but the $n = 10$ intervals are narrower, and narrower is better**: a more precise estimate of $\mu$ at the same probability of being right. There is a second, subtler bonus: with larger samples the sample means themselves sit closer to $\mu$ on average, which is exactly *why* the smaller margins still capture it.

$$
\text{margin of error} = z \cdot \frac{\sigma}{\sqrt{n}} \quad\text{at a fixed confidence level: } \; n \uparrow \;\Rightarrow\; \text{interval} \downarrow
$$

### Dial 2: the confidence level

Keep $n = 1$, so the curve of $\bar{X}$ is fixed — and settle for a **70%** confidence level instead of 95%. The distribution doesn't change at all; you simply draw smaller margins, because the sample mean now only needs to land inside them 70% of the time. Generate many intervals of each kind: the 95% batch captures $\mu$ about 95% of the time; the 70% batch only about 70% of the time, missing in 30% of cases.

The important non-difference: the sample means on the 70% side are **not** farther from $\mu$ — they come from the same curve. You just chose smaller margins, so more intervals miss. This is the lost-key trade-off again: a higher probability of containing the key demands a longer search distance.

### The summary picture

- A confidence interval has **two ingredients**: a randomly generated sample mean, and a margin of error added to each side. Because the sample mean is up to chance, the interval is random too.
- The **confidence level** is the probability that an interval generated this way contains $\mu$ (typically 95%) — and you will never be able to confirm whether *yours* did, since $\mu$ stays unknown.
- The dream is a **high confidence level and a narrow interval** — 99% sure the truth is in a tiny range. You can't have both for free.
- The honest dial is **more data**: a larger sample shrinks the sampling distribution's variance, letting smaller margins capture the same share of it. Lowering the confidence level also narrows the interval, but levels below 90% are rarely used — at some point, more accuracy simply requires more data.

## 3. Worked Examples

Throughout, Statistopia heights: $\sigma = 10$ cm, and $z_{95\%} \approx 1.96$.

### Example 1: the margin at three sample sizes

$$
n = 1:\;\; 1.96 \cdot \tfrac{10}{\sqrt{1}} = 19.6 \text{ cm} \qquad
n = 2:\;\; 1.96 \cdot \tfrac{10}{\sqrt{2}} \approx 13.9 \text{ cm} \qquad
n = 10:\;\; 1.96 \cdot \tfrac{10}{\sqrt{10}} \approx 6.2 \text{ cm}
$$

Ten people instead of one cut the search distance from 19.6 cm to 6.2 cm — a factor of $\sqrt{10} \approx 3.16$ — with no loss of confidence.

### Example 2: two ways to halve an interval

Starting at $n = 1$, 95%, margin 19.6 cm:

- **With data:** margin $\propto 1/\sqrt{n}$, so halving it needs $n = 4$ — three more people, still 95% confident.
- **With risk:** halving the margin at $n = 1$ means halving $z$ from 1.96 to 0.98 — and a standard normal lands within $\pm 0.98$ only about **67%** of the time. Same width, but now a third of such intervals miss $\mu$.

### Example 3: how much data does a target width need?

For a margin of at most $m$, solve $z\sigma/\sqrt{n} \le m$ for $n$:

$$
n \;\ge\; \left(\frac{z\,\sigma}{m}\right)^{\!2}
$$

A 2 cm margin at 95%: $n \ge (1.96 \cdot 10 / 2)^2 = 96.04$, so 97 people. Note the square: **10× the precision costs 100× the data** — precision is expensive, which is why the exercises call this the tyranny of $\sqrt{n}$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project) and the quantile helper from the last notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(12)

MU_TRUE = 170.0        # Statistopia's mean height (cm) -- fixed, "unknown"
SIGMA = 10.0           # population standard deviation, known

def z_for_confidence(confidence):
    """z such that a standard normal lands in [-z, z] with the given probability."""
    return float(sp.sqrt(2) * sp.erfinv(sp.Float(confidence)))

def margin_of_error(confidence, n):
    return z_for_confidence(confidence) * SIGMA / np.sqrt(n)

### The margin table: both dials at once

In [ ]:
print(f"{'':>6s}" + "".join(f"{f'{lvl:.0%} level':>13s}" for lvl in (0.70, 0.90, 0.95, 0.99)))
for n in (1, 2, 10, 100):
    row = "".join(f"{margin_of_error(lvl, n):>13.1f}" for lvl in (0.70, 0.90, 0.95, 0.99))
    print(f"n={n:<4d}{row}")
print("\n(margins of error in cm: down a column data helps, left along a row risk helps)")

### Both dials, checked empirically

Each recipe's promise: coverage equals the confidence level, whatever the sample size. Simulate 10,000 samples per configuration — drawing each sample mean from $N(\mu, \sigma^2/n)$ — and compare.

In [ ]:
print(f"{'n':>4s} {'level':>7s} {'margin':>8s} {'coverage':>9s}")
for n, level in [(1, 0.95), (2, 0.95), (10, 0.95), (1, 0.70)]:
    m = margin_of_error(level, n)
    xbars = rng.normal(MU_TRUE, SIGMA / np.sqrt(n), size=10_000)
    coverage = (np.abs(xbars - MU_TRUE) <= m).mean()
    print(f"{n:>4d} {level:>7.0%} {m:>7.1f}  {coverage:>9.1%}")
print("\nCoverage tracks the level, never the sample size: n buys narrowness, not correctness.")

## 5. Visualizations

### Taller and skinnier: the curve at $n$ = 1, 2, 10

The center never moves; the spread shrinks by $\sqrt{n}$, and the 95% margins (dotted) shrink with it.

In [ ]:
heights = np.linspace(MU_TRUE - 35, MU_TRUE + 35, 600)

plt.figure(figsize=(10, 5))
for n, color in [(1, "tab:blue"), (2, "tab:orange"), (10, "tab:green")]:
    sd = SIGMA / np.sqrt(n)
    density = np.exp(-0.5 * ((heights - MU_TRUE) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    m = margin_of_error(0.95, n)
    plt.plot(heights, density, color=color, linewidth=2,
             label=f"n = {n}:  $\\sigma_{{\\bar{{X}}}}$ = {sd:.1f} cm,  95% margin = {m:.1f} cm")
    for x in (MU_TRUE - m, MU_TRUE + m):
        plt.axvline(x, color=color, linestyle=":", linewidth=1.2, alpha=0.8)
plt.axvline(MU_TRUE, color="black", linewidth=1)
plt.xlabel(r"sample mean $\bar{x}$ (cm)")
plt.ylabel("density")
plt.title(r"More data: the curve of $\bar{X}$ gets taller and skinnier, and the margins move in")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Dial 1 in action: 95% intervals at $n$ = 2 vs $n$ = 10

Fifty intervals per panel. Both panels are ~95% green — but the right panel's intervals are $\sqrt{5}$ times shorter. Same confidence, more precision.

In [ ]:
def interval_panel(ax, n, level, n_intervals=50):
    m = margin_of_error(level, n)
    means = rng.normal(MU_TRUE, SIGMA / np.sqrt(n), size=n_intervals)
    caught = np.abs(means - MU_TRUE) <= m
    for i, (xb, ok) in enumerate(zip(means, caught)):
        color = "tab:green" if ok else "tab:red"
        ax.plot([xb - m, xb + m], [i, i], color=color, linewidth=1.8, alpha=0.85)
        ax.plot([xb], [i], marker="o", color=color, markersize=2.5)
    ax.axvline(MU_TRUE, color="black", linestyle="--", linewidth=1.3)
    ax.set_xlim(MU_TRUE - 32, MU_TRUE + 32)
    ax.set_title(f"n = {n}, level = {level:.0%}: "
                 f"{caught.sum()}/{n_intervals} contain $\\mu$", fontsize=10)
    ax.set_xlabel("height (cm)")
    ax.grid(True, alpha=0.3, axis="x")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6), sharey=True)
interval_panel(axes[0], n=2, level=0.95)
interval_panel(axes[1], n=10, level=0.95)
axes[0].set_ylabel("interval number")
plt.suptitle("Same 95% promise -- but the larger sample keeps it with far narrower intervals")
plt.tight_layout()
plt.show()

### Dial 2 in action: 95% vs 70% at $n$ = 1

Same sampling distribution in both panels — the sample means are no farther from $\mu$ on the right. The intervals are just narrower by choice, so more of them miss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6), sharey=True)
interval_panel(axes[0], n=1, level=0.95)
interval_panel(axes[1], n=1, level=0.70)
axes[0].set_ylabel("interval number")
plt.suptitle("Lowering the level narrows the intervals -- and pays for it in red")
plt.tight_layout()
plt.show()

### The mock simulator: each dial swept alone

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

n_axis = np.arange(1, 101)
axes[0].plot(n_axis, [margin_of_error(0.95, n) for n in n_axis],
             color="tab:green", linewidth=2)
axes[0].set_xlabel("sample size n   (confidence fixed at 95%)")
axes[0].set_ylabel("margin of error (cm)")
axes[0].set_title(r"Dial 1: margin $= 1.96\,\sigma/\sqrt{n}$", fontsize=10)
axes[0].grid(True, alpha=0.3)

levels = np.linspace(0.50, 0.995, 200)
axes[1].plot(levels * 100, [margin_of_error(lvl, 1) for lvl in levels],
             color="tab:purple", linewidth=2)
for lvl in (0.70, 0.95):
    axes[1].plot(lvl * 100, margin_of_error(lvl, 1), "ko", markersize=4)
    axes[1].annotate(f"{lvl:.0%}: {margin_of_error(lvl, 1):.1f} cm",
                     xy=(lvl * 100, margin_of_error(lvl, 1)),
                     xytext=(-8, 8), textcoords="offset points",
                     ha="right", fontsize=9)
axes[1].set_xlabel("confidence level (%)   (n fixed at 1)")
axes[1].set_ylabel("margin of error (cm)")
axes[1].set_title("Dial 2: margin grows with the promised confidence", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle("The two dials of the confidence interval")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: the sample-size dial, the confidence dial, the full margin table, and why coverage follows the level, not n.

<img src="img/02_two_dials.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** With $\sigma = 10$ and $n = 25$, compute the margin of error at the 90%, 95%, and 99% levels. Then state what happens to each margin if $n$ grows to 100.

**Exercise 2.** True or false, with a one-sentence reason: "In the 70% panels, the sample means tend to be farther from $\mu$ than in the 95% panels."

### Intermediate

**Exercise 3.** A colleague at $n = 1$, $\sigma = 10$ insists on using a margin of exactly 10 cm. What confidence level are they actually working at? What level would the same 10 cm margin deliver at $n = 4$?

**Exercise 4.** You need the 95% margin of error to be at most 2 cm ($\sigma = 10$). Find the required $n$, then verify by simulating that recipe's coverage and width.

### Advanced

**Exercise 5.** *Coverage is level-only.* Over the grid $n \in \{1, 5, 25\}$ × level $\in \{70\%, 90\%, 95\%\}$, simulate 20,000 intervals per cell and tabulate empirical coverage and mean width. Confirm coverage depends only on the level while width scales as $z/\sqrt{n}$.

**Exercise 6.** *The tyranny of $\sqrt{n}$.* Plot the sample size required for a 95% margin of $m$ cm as $m$ shrinks from 10 to 0.5. Confirm the 10×-precision-costs-100×-data rule, and use the curve to explain why, past some point, narrowing an interval by lowering the confidence level stops being a bargain compared to collecting data.

## 7. Solutions

In [ ]:
# Exercise 1: margins at n = 25, then n = 100
for level in (0.90, 0.95, 0.99):
    m25, m100 = margin_of_error(level, 25), margin_of_error(level, 100)
    print(f"{level:.0%}: margin at n=25 is {m25:.2f} cm; at n=100 it is "
          f"{m100:.2f} cm (halved: 4x the data, 2x the precision)")

In [ ]:
# Exercise 2: false -- same sampling distribution
print("Exercise 2: False. At fixed n the sample means come from the same")
print("N(mu, sigma^2/n) curve regardless of the chosen level; the 70% intervals")
print("miss more often only because their margins were drawn smaller.")

In [ ]:
# Exercise 3: what level does a 10 cm margin buy?
def confidence_for_margin(m, n):
    z = m * np.sqrt(n) / SIGMA
    return float(sp.erf(z / sp.sqrt(2)))       # P(|Z| <= z) for a standard normal

print(f"Exercise 3: 10 cm at n=1 -> z = 1.0 -> level = "
      f"{confidence_for_margin(10, 1):.1%}")
print(f"            10 cm at n=4 -> z = 2.0 -> level = "
      f"{confidence_for_margin(10, 4):.1%}")
print("Same width, wildly different promises -- the sample size did the work.")

In [ ]:
# Exercise 4: n for a 2 cm margin at 95%
z95 = z_for_confidence(0.95)
n_required = int(np.ceil((z95 * SIGMA / 2) ** 2))
m_actual = margin_of_error(0.95, n_required)
xbars = rng.normal(MU_TRUE, SIGMA / np.sqrt(n_required), size=20_000)
coverage = (np.abs(xbars - MU_TRUE) <= m_actual).mean()
print(f"Exercise 4: n >= (1.96 * 10 / 2)^2 = {(z95 * SIGMA / 2) ** 2:.2f} "
      f"-> n = {n_required}")
print(f"check: margin = {m_actual:.2f} cm, simulated coverage = {coverage:.1%}")

In [ ]:
# Exercise 5: coverage depends only on the level; width only adds 1/sqrt(n)
print(f"{'n':>4s} {'level':>7s} {'coverage':>9s} {'width (cm)':>11s} {'z/sqrt(n)':>10s}")
for n in (1, 5, 25):
    for level in (0.70, 0.90, 0.95):
        m = margin_of_error(level, n)
        xbars = rng.normal(MU_TRUE, SIGMA / np.sqrt(n), size=20_000)
        cov = (np.abs(xbars - MU_TRUE) <= m).mean()
        print(f"{n:>4d} {level:>7.0%} {cov:>9.1%} {2 * m:>11.1f} "
              f"{z_for_confidence(level) / np.sqrt(n):>10.2f}")
print("\nEach level's coverage column repeats across n; width is 2 sigma z/sqrt(n).")

In [ ]:
# Exercise 6: required n vs target margin
targets = np.linspace(0.5, 10, 200)
n_needed = (z95 * SIGMA / targets) ** 2

plt.figure(figsize=(9, 4.4))
plt.plot(targets, n_needed, color="tab:red", linewidth=2)
plt.yscale("log")
for m in (10, 2, 1, 0.5):
    n_m = (z95 * SIGMA / m) ** 2
    plt.plot(m, n_m, "ko", markersize=4)
    plt.annotate(f"m = {m} cm: n = {n_m:,.0f}", xy=(m, n_m),
                 xytext=(6, 6), textcoords="offset points", fontsize=9)
plt.xlabel("target 95% margin of error m (cm)")
plt.ylabel("required sample size n (log scale)")
plt.title(r"Exercise 6: n = $(1.96\,\sigma/m)^2$ -- 10x the precision costs 100x the data")
plt.grid(True, alpha=0.3)
plt.show()

shrink = 1 - z_for_confidence(0.50) / z95
n_equiv = (z95 / z_for_confidence(0.50)) ** 2
print("Halving m quadruples n, so precision gets expensive fast. But the risk dial")
print(f"is bounded: dropping all the way from 95% to a coin-flip 50% promise shrinks")
print(f"the margin by {shrink:.0%} -- the same narrowing that {n_equiv:.0f}x the data buys")
print("honestly -- and there the dial ends. Only the data dial keeps turning.")

## 8. Conclusion

- $\bar{X}$ is centered on $\mu$ at every sample size; only its spread changes: $\sigma_{\bar{X}} = \sigma/\sqrt{n}$. More data makes the curve taller and skinnier without moving it.
- **Dial 1 — sample size:** at a fixed confidence level, the margin of error $z\sigma/\sqrt{n}$ shrinks with $n$. Coverage stays at the promised level; the intervals just get narrower — strictly better.
- **Dial 2 — confidence level:** at a fixed $n$, choosing 70% instead of 95% shrinks the margins but makes 30% of intervals miss. The sample means are no worse; the promise is.
- Confidence intervals have two ingredients — a random sample mean plus symmetric margins — so the interval is random, its confidence level is the probability the *recipe* captures $\mu$, and you never learn whether yours did.
- You can't have a narrow interval and high confidence for free: collect more data (the honest dial, cost $\propto 1/m^2$) or accept more risk (rarely worth it — levels below 90% are seldom used).

Next: how the margins of error — and therefore confidence intervals — are actually calculated.